[Guide Index](README.md)


# 06b Choosing Where Your Data Lives

Most examples in the StarLayer user guide demonstrate the functionality of `StarLayerGraph` using the default, in-process graph. Real applications often need data to live somewhere else, or to survive a restart.

Two independent choices matter here - **Transport** (does the store live in this process, or on a separate server reached over the network) and **Persistence** (does it write to disk, or stay in memory) - see [`starlayergraph.md`](../../packages/starlayer/docs/graph/starlayergraph.md)'s own "Terminology" section for the full picture, including a third, separate axis (**Representation** - `rdf-1.1` vs. `rdf-1.2`, the `backend=` parameter) that this guide doesn't change.

`StarLayerGraph` supports three ways to reach a different Transport/Persistence combination:
- **A SPARQL endpoint** - remote.
- **A graph store** - a third-party rdflib `Store` plugin (e.g. SQLAlchemy/SQLite) - in-process, persisted.
- **An embedded graph engine** - in-process, with genuine native RDF 1.2 triple-term storage available too.

The examples below demonstrate how to configure each.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later cells reuse variables from earlier ones.

Section 1 needs a running Oxigraph and/or Fuseki instance (each example skips cleanly if its endpoint isn't reachable) — every example clears all existing data in the dataset it connects to (`CLEAR ALL` / equivalent), so don't point one at a database you care about. Section 2 needs the `rdflib-sqlalchemy` package, installed inline by that section's own first cell. Section 3 needs the optional `pyoxigraph` package (`pip install starlayer.graph[oxigraph-embedded]`).


In [1]:
from starlayer.graph import ENTAILMENT, StarLayerGraph, Namespace, TripleTerm

EX = Namespace("http://example.org/")

## 1. Using a SPARQL endpoint

`StarLayerGraph` can work against any SPARQL endpoint via a `SPARQLUpdateStore`.  `StarLayerGraph` has been tested against both Oxigraph and Fuseki.  Both support the RDF 1.2 and RDF 1.1 representations.

When possible, the RDF 1.2 representation is a better choice when working with RDF 1.2 graphs. The RDF 1.1 representation is useful when integrating RDF 1.2 data with an existing SPARQL graph that operates using RDF 1.1.

The following examples require a running instance of Oxigraph and/or Fuseki.  Initiating a session is outside the scope of this user guide.

The examples assume the following end-points.  Your end-points may differ.
- Oxigraph: `http://localhost:7878/`
- Fuseki: `http://localhost:3030/starlayergraph` (5.5+ needed for the native RDF 1.2 representation)


In [2]:

import urllib.error
import requests
from rdflib.plugins.stores.sparqlstore import SPARQLUpdateStore

def demo_sparql_endpoint(label, query_endpoint, update_endpoint, auth=None, backend="rdf-1.2"):
    """Connect, clear, write a reification, and print the resulting graph."""
    def make_store():
        return SPARQLUpdateStore(query_endpoint=query_endpoint, update_endpoint=update_endpoint, auth=auth)

    try:
        g = StarLayerGraph(store=make_store(), identifier=EX.main, backend=backend)
        g.update("CLEAR ALL")
    except (urllib.error.URLError, requests.exceptions.ConnectionError):
        # backend="rdf-1.1" goes through plain rdflib's own SPARQLUpdateStore,
        # which raises urllib.error.URLError when unreachable; backend="rdf-1.2"
        # (native) goes through starlayer's own HTTP layer, which uses
        # requests directly (backends/native.py) and raises
        # requests.exceptions.ConnectionError instead - both are caught here
        # so this guard actually protects both call shapes below, not just one.
        print(f"{label}: skipping - no live instance at {query_endpoint}")
        return

    g.bind("ex", EX)
    g.add_reification(EX.claim, TripleTerm(EX.bob, EX.knows, EX.carol))
    print(f"{label}:")
    print(g.serialize())

# test oxigraph
demo_sparql_endpoint(
    "Oxigraph",
    query_endpoint="http://localhost:7878/query",
    update_endpoint="http://localhost:7878/update",
)

# test fueski
demo_sparql_endpoint(
    "Fuseki",
    query_endpoint="http://localhost:3030/starlayergraph/query",
    update_endpoint="http://localhost:3030/starlayergraph/update",
    auth=("admin", "admin"),
)

Oxigraph:
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> .

Fuseki:
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> .



### 1.1 RDF 1.1 SPARQL endpoint

When operating with an RDF 1.1 SPARQL end-point, change the backend to "rdf-1.1".  

Note that RDF 1.1 mode handles RDF 1.2 terms, such as triple terms.

In [3]:
demo_sparql_endpoint(
    "Oxigraph, rdf-1.1 mode",
    query_endpoint="http://localhost:7878/query",
    update_endpoint="http://localhost:7878/update",
    backend="rdf-1.1",
)

Oxigraph, rdf-1.1 mode:
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> .



### 1.2 Reasoning-enabled Fuseki (`entailment="native"`)

`query(..., entailment="native")` (only legal with `backend="rdf-1.2"`) changes nothing about how the query is sent - it's a self-documenting label, since the native path already passes queries straight through unmodified. (This "native" is unrelated to the native/`rdf-1.2` Representation terminology elsewhere in this guide - it's just this parameter's own label for "let the endpoint's own reasoning configuration decide.") Entailment support is entirely a property of how the *endpoint itself* is configured. See [fuseki-reasoning-setup.md](../../packages/starlayer/docs/graph/fuseki-reasoning-setup.md) for how to stand up a Fuseki instance that reasons before answering - this example expects one at `localhost:3031`, a dataset named `starlayergraph-rdfs`, separate from the plain one above, started with `--rdfs=schema.ttl` declaring `ex:Manager rdfs:subClassOf ex:Employee`. See the [Graph Inferencing guide](02b-graphs-inferencing.ipynb) for `entailment=ENTAILMENT.RDFS`/`ENTAILMENT['OWL-RDF-Based']` - `StarLayerGraph`'s own reasoning, which needs none of this setup.

In [4]:
try:
    store = SPARQLUpdateStore(
        query_endpoint="http://localhost:3031/starlayergraph-rdfs/query",
        update_endpoint="http://localhost:3031/starlayergraph-rdfs/update",
    )
    g_rdfs = StarLayerGraph(store=store, backend="rdf-1.2")
    g_rdfs.update("CLEAR ALL")
except (urllib.error.URLError, requests.exceptions.ConnectionError):
    print("skipping - no live reasoning-enabled instance at localhost:3031")
else:
    g_rdfs.bind("ex", EX)
    g_rdfs.update("PREFIX ex: <http://example.org/> INSERT DATA { ex:alice a ex:Manager }")

    rows = list(g_rdfs.query(
        "PREFIX ex: <http://example.org/> SELECT ?x WHERE { ?x a ex:Employee }",
        entailment="native",
    ))
    print("alice is an Employee (via rdfs:subClassOf, reasoned by Fuseki itself):", [str(r.x) for r in rows])

skipping - no live reasoning-enabled instance at localhost:3031


## 2. Using a graph store

Any rdflib `Store` plugin works transparently with `StarLayerGraph` - an in-process Transport choice (SQLAlchemy/SQLite below) that adds Persistence without touching Representation: a generic `Store` plugin carries no triple-term concept of its own, so data still round-trips via the default `rdf-1.1` `tt:HASH` encoding regardless.

In [5]:
!pip install -q rdflib-sqlalchemy

import tempfile
import warnings

with warnings.catch_warnings():
    # rdflib-sqlalchemy's own import warns about its (still-working) use of
    # the deprecated pkg_resources API - not something this guide is concerned with.
    warnings.filterwarnings("ignore", category=UserWarning, module="rdflib_sqlalchemy")
    import rdflib_sqlalchemy
rdflib_sqlalchemy.registerplugins()

db_path = tempfile.mktemp(suffix=".sqlite")
uri = f"sqlite:///{db_path}"

writer = StarLayerGraph(store="SQLAlchemy", identifier=EX.main)
writer.open(uri, create=True)
writer.bind("ex", EX)
writer.add_reification(EX.claim, TripleTerm(EX.bob, EX.knows, EX.carol))
writer.commit()
writer.close()

# fresh graph object, same database file - proves the data actually persisted
reader = StarLayerGraph(store="SQLAlchemy", identifier=EX.main)
reader.open(uri, create=False)
reader.bind("ex", EX)
print(reader.serialize())
reader.close()

@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> .



## 3. Using an embedded graph engine

Oxigraph (the same engine behind section 1's SPARQL-endpoint examples) also ships as `pyoxigraph`, a Python extension that embeds its Rust engine in-process - no server, no HTTP round trip (a different Transport from section 1's remote examples). `embedded_oxigraph_store()` wraps it as a `store=` for `StarLayerGraph`, combined with `backend="rdf-1.2"` to get real native RDF 1.2 triple-term storage (not this project's own `tt:HASH` encoding, the same encoding the default in-process store uses) with zero network overhead.

This is different from passing an ordinary third-party rdflib `Store` plugin (section 2's pattern, or `oxrdflib`'s own `Store` wrapping the same `pyoxigraph` library): a generic `Store` plugin only ever gets `StarLayerGraph`'s default `tt:HASH`-encoded behavior, since rdflib's own `Store` protocol has no concept of a triple term at all. `backend="rdf-1.2"` bypasses that protocol entirely for genuine native storage - see [`starlayer.graph.backends.native_pyoxigraph`](../../packages/starlayer/starlayer/graph/backends/native_pyoxigraph.py).

In [6]:
from starlayer.graph import embedded_oxigraph_store

g = StarLayerGraph(store=embedded_oxigraph_store(), backend="rdf-1.2")
g.bind("ex", EX)
g.add_reification(EX.claim, TripleTerm(EX.bob, EX.knows, EX.carol))
print(g.serialize())
g.close()

@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> .



### 3.1 Setting this as the process-global default

A developer who always wants Oxigraph in-process, native RDF 1.2 storage can set it once, process-wide, with `set_default_representation()` - every bare `StarLayerGraph()`/`StarLayerDataset()` constructed afterward (no explicit `store=`/`backend=`) picks it up. This is a plain, permanent global by design (not a scoped context manager) - meant for "a developer configures this once for their whole application," not for switching representations mid-process. An explicit `store=`/`backend=` on any individual call always overrides it.

In [7]:
from starlayer.graph import set_default_representation

set_default_representation(embedded_oxigraph_store)

g_default = StarLayerGraph()  # no store=/backend= given at all
print("is_native:", g_default._is_native)
g_default.close()

set_default_representation(None)  # restore the ordinary default for the rest of this guide

is_native: True
